# Pipeline Klasifikasi Tutupan Lahan di Jawa Timur

## Business Understanding
**Tujuan:** Memetakan permukaan bumi menggunakan citra satelit untuk mendapatkan informasi tata ruang dan tutupan lahan yang akurat sebagai dasar pengambilan keputusan.

**Konteks & Urgensi:** Pertumbuhan penduduk dan pembangunan di Jawa Timur berdampak langsung pada perubahan fungsi lahan. Pemantauan tutupan lahan secara berkala sangat krusial untuk:
- Mencegah alih fungsi lahan pertanian yang mengancam ketahanan pangan.
- Memantau deforestasi dan degradasi hutan, khususnya hutan mangrove yang berfungsi sebagai sabuk hijau pelindung pantai.
- Mendukung perencanaan tata ruang wilayah (RTRW) yang berkelanjutan dan berbasis data.

In [1]:
import numpy as np
import rasterio

def process_sentinel2_data(b4_red, b8a_nir, b11_swir):
    """
    Mengekstrak indeks spektral dan membedakan Hutan Biasa vs Hutan Mangrove
    menggunakan kombinasi Band 8A (NIR Narrow), Band 11 (SWIR), dan Band 4 (Red)
    dari citra satelit Sentinel-2.
    """
    # Menghindari pembagian dengan nol (zero division)
    b4_red = np.where(b4_red == 0, 1e-10, b4_red)
    b8a_nir = np.where(b8a_nir == 0, 1e-10, b8a_nir)
    b11_swir = np.where(b11_swir == 0, 1e-10, b11_swir)

    # Menghitung Normalized Difference Vegetation Index (NDVI) -> Deteksi Kerapatan Vegetasi
    ndvi = (b8a_nir - b4_red) / (b8a_nir + b4_red)

    # Menghitung Land Surface Water Index (LSWI) / NDMI -> Deteksi Kelembaban/Genangan Air
    lswi = (b8a_nir - b11_swir) / (b8a_nir + b11_swir)

    # Logika pembedaan (berdasarkan karakteristik spektral):
    # Hutan biasa memiliki NDVI tinggi (vegetasi lebat) dan LSWI moderat/rendah (lebih kering).
    # Hutan Mangrove memiliki NDVI tinggi DAN LSWI tinggi (karena habitatnya di area pasang surut/perairan).
    mangrove_mask = (ndvi > 0.6) & (lswi > 0.3)
    forest_mask = (ndvi > 0.6) & (lswi <= 0.3)

    return ndvi, lswi, mangrove_mask, forest_mask

print("Modul pemrosesan data Sentinel-2 berhasil diinisiasi.")

Modul pemrosesan data Sentinel-2 berhasil diinisiasi.


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Inisialisasi Pipeline Machine Learning untuk klasifikasi dataset berskala besar
# Target Prediksi (5 Kelas): 
# 1: Bangunan (Built-up)
# 2: Lahan Pertanian (Farm land)
# 3: Perairan (Water body)
# 4: Hutan (Forest)
# 5: Hutan Mangrove

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        n_jobs=-1,  # n_jobs=-1 memastikan penggunaan seluruh core CPU secara paralel
        random_state=42,
        class_weight='balanced'
    ))
])

# ---- Bagian ini hanya simulasi data training agar pipeline bisa dijalankan ----
# Misal fitur input kita: [Band2, Band3, Band4, Band8A, Band11, NDVI, LSWI]
X_dummy = np.random.rand(2000, 7)
y_dummy = np.random.randint(1, 6, 2000)

X_train, X_test, y_train, y_test = train_test_split(X_dummy, y_dummy, test_size=0.2, random_state=42)
# ------------------------------------------------------------------------------

print("Melatih model Random Forest...")
rf_pipeline.fit(X_train, y_train)
score = rf_pipeline.score(X_test, y_test)

print(f"Akurasi model pada data validasi: {score:.2f}")
print("Pipeline Machine Learning siap untuk memprediksi citra tutupan lahan yang sesungguhnya.")

Melatih model Random Forest...
Akurasi model pada data validasi: 0.20
Pipeline Machine Learning siap untuk memprediksi citra tutupan lahan yang sesungguhnya.


In [3]:
import folium
import json
import os

# Membuat peta dasar (Basemap) yang berpusat di area Jawa Timur
m = folium.Map(location=[-7.7, 112.5], zoom_start=8, tiles=None)

# Menambahkan Basemap Satelit ESRI World Imagery
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satelit (Esri)',
    overlay=False,
    control=True
).add_to(m)

# Pemetaan ID Kelas ke Nama Layer (untuk filter interaktif)
kelas_mapping = {
    1: "Bangunan (Built-up)",
    2: "Lahan Pertanian (Farm land)",
    3: "Perairan (Water body)",
    4: "Hutan (Forest)",
    5: "Hutan Mangrove"
}

# Membuat FeatureGroup (grup layer) terpisah untuk setiap kelas tutupan lahan
feature_groups = {nama: folium.FeatureGroup(name=nama) for nama in kelas_mapping.values()}

# Fungsi styling: menampilkan poligon digitasi dengan warna biru transparan
def style_function(feature):
    return {
        'fillColor': 'blue',
        'color': 'blue',
        'weight': 2,
        'fillOpacity': 0.4
    }

geojson_path = "data_hasil_digitasi.geojson"

# Membaca file GeoJSON lokal dan memisahkannya ke masing-masing FeatureGroup
if os.path.exists(geojson_path):
    with open(geojson_path, 'r') as f:
        geo_data = json.load(f)
        
    for feature in geo_data.get('features', []):
        # Asumsi informasi kelas tersimpan di properti 'class_id' pada GeoJSON
        class_id = feature.get('properties', {}).get('class_id')
        layer_name = kelas_mapping.get(class_id, "Lainnya")
        
        # Tambahkan ke grup layer spesifik
        if layer_name in feature_groups:
            folium.GeoJson(
                feature,
                style_function=style_function,
                tooltip=layer_name # Menampilkan popup hover nama kelas
            ).add_to(feature_groups[layer_name])
else:
    print(f"Peringatan: File lokal '{geojson_path}' belum tersedia. Menampilkan peta satelit tanpa overlay poligon.")

# Menambahkan semua grup layer (FeatureGroups) ke dalam peta utama
for fg in feature_groups.values():
    fg.add_to(m)

# Menambahkan Layer Control (Checkbox Interaktif)
# Checkbox ini yang memungkinkan user untuk mencentang/menghilangkan poligon sesuai kelasnya
folium.LayerControl(collapsed=False).add_to(m)

# Menampilkan hasil render peta
m

ModuleNotFoundError: No module named 'folium'